In [36]:
import os
import pandas as pd

current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")


def df_to_latex(df):
    num_columns = len(df.columns)
    latex_table = df.to_latex(
        index=False,  # To not include the DataFrame index as a column in the table
        caption="CAPTION",  # The caption to appear above the table in the LaTeX document
        label="tab:LABEL",  # A label used for referencing the table within the LaTeX document
        position="htbp",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
        column_format="l"+(num_columns-1)*"c",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
        escape=True,  # escape LaTeX special characters in the DataFrame
        float_format="{:0.2f}".format  # Formats floats to two decimal places
    )
    print(latex_table)

def df_to_latex(df, caption="CAPTION", label=None):
    num_columns = len(df.columns)
    
    latex_table = df.to_latex(
        index=False,
        caption=caption,
        label="tab:"+caption.replace(" ", "_").lower() if label is None else label,
        position="htbp",
        column_format="l" + (num_columns - 1) * "c",
        escape=True,
        float_format=lambda x: "{:.3f}".format(x).lstrip("0") if 0 < x < 1 else "{:.2f}".format(x)
    )

    lines = latex_table.splitlines()

        # --- Modify the header row (second line in the table) ---
    header_line_index = next(i for i, line in enumerate(lines) if '&' in line and '\\' in line)
    header_line = lines[header_line_index]

    header_cells = header_line.split('&')
    
    # Remove \\ from the last cell before wrapping
    if header_cells[-1].strip().endswith(r'\\'):
        header_cells[-1] = header_cells[-1].strip()[:-2].strip()  # remove \\

    header_cells = [f"\\texttt{{{cell.strip()}}}" for cell in header_cells]
    lines[header_line_index] = ' & '.join(header_cells) + r" \\"


    # --- Modify the first column of each row (data rows only) ---
    for i in range(header_line_index + 2, len(lines)):
        if '&' in lines[i]:
            parts = lines[i].split('&')
            parts[0] = f"\\texttt{{{parts[0].strip()}}}"  # First column only
            lines[i] = ' & '.join(parts)

    final_latex = '\n'.join(lines)
    print(final_latex)


In [37]:
path = "./src/agnews/experiments/RoBERTA_500samples/RoBERTA_results_dev.csv"
df = pd.read_csv(path)
df = df.astype({"train_time":"int", "eval_time":"int"})
df_to_latex(df, caption="RoBERTA")

\begin{table}[htbp]
\caption{RoBERTA}
\label{tab:roberta}
\begin{tabular}{lccccc}
\toprule
\texttt{method} & \texttt{f1\_micro} & \texttt{f1\_macro} & \texttt{accuracy} & \texttt{train\_time} & \texttt{eval\_time} \\
\midrule
\texttt{500real} &  .924  &  .885  &  .924  &  317  &  8 \\
\texttt{250real\_250generic} &  .912  &  .864  &  .912  &  312  &  8 \\
\texttt{250real\_250targeted} &  .924  &  .887  &  .924  &  311  &  8 \\
\texttt{50real\_450generic} &  .886  &  .839  &  .886  &  318  &  8 \\
\texttt{50real\_450targeted} &  .890  &  .826  &  .890  &  315  &  8 \\
\texttt{500generic} &  .762  &  .724  &  .762  &  319  &  8 \\
\texttt{500targeted} &  .852  &  .806  &  .852  &  310  &  8 \\
\bottomrule
\end{tabular}
\end{table}


In [38]:
path = "./src/agnews/experiments/fewshotCasualLM_500samples/fewshotCasualLM_results_dev.csv"
df = pd.read_csv(path)
df = df.astype({"eval_time":"int"})
df_to_latex(df, caption="fewshot CasualLM")

\begin{table}[htbp]
\caption{fewshot CasualLM}
\label{tab:fewshot_casuallm}
\begin{tabular}{lcccc}
\toprule
\texttt{method} & \texttt{f1\_micro} & \texttt{f1\_macro} & \texttt{accuracy} & \texttt{eval\_time} \\
\midrule
\texttt{deepseek\_1ex4label} &  .450  &  .313  &  .450  &  99 \\
\texttt{deepseek\_2ex4label} &  .468  &  .458  &  .468  &  105 \\
\texttt{llama2\_1ex4label} &  .692  &  .537  &  .692  &  234 \\
\texttt{llama2\_2ex4label} &  .836  &  .462  &  .836  &  288 \\
\bottomrule
\end{tabular}
\end{table}


In [39]:
path = "./src/agnews/experiments/adaptiveICL_1000samples/adaptiveICL_results_test.csv"
df = pd.read_csv(path)
df = df.astype({"prepare_time":"int","eval_time":"int"})
df_to_latex(df, caption="Adaptive ICL")

\begin{table}[htbp]
\caption{Adaptive ICL}
\label{tab:adaptive_icl}
\begin{tabular}{lccccc}
\toprule
\texttt{name} & \texttt{f1\_micro} & \texttt{f1\_macro} & \texttt{accuracy} & \texttt{prepare\_time} & \texttt{eval\_time} \\
\midrule
\texttt{llama2\_1000real} &  .764  &  .730  &  .764  &  3954  &  277 \\
\texttt{llama2\_500real+500syn} &  .720  &  .684  &  .720  &  3339  &  272 \\
\texttt{llama2\_1000syn} &  .496  &  .468  &  .496  &  2990  &  222 \\
\bottomrule
\end{tabular}
\end{table}
